In [37]:
import numpy as np
from tqdm import tqdm
import scipy.linalg as la
import numpy.linalg as npla

import torch
import normflows as nf

import pid
import pid.utils.distributions as dist
from pid.utils.generate import sample_mult_poisson

In [38]:
def get_params():
    lamda_m = np.array([2., 2.])
    lamda_x = np.array([1.,])      # Noise to be added to X: shape (d_X,)
    lamda_y = np.array([1.,])

    w_x1_vals = np.arange(11) / 10
    w_x2 = 0.5
    w_y1 = 0.5
    w_y2 = 0.5

    return lamda_m, lamda_x, lamda_y, w_x1_vals, w_x2, w_y1, w_y2


def pid_flows(base, n_flows, latent_dim, flow_type='Spline'):
    flows = []
    for i in range(n_flows):
        if flow_type == 'Spline':
            flows += [nf.flows.AutoregressiveRationalQuadraticSpline(latent_dim, 2, 128)]
            flows += [nf.flows.LULinearPermute(latent_dim)]
        elif flow_type == 'Planar':
            flows += [nf.flows.Planar((latent_dim,))]
        elif flow_type == 'Radial':
            flows += [nf.flows.Radial((latent_dim,))]
        else:
            raise NotImplementedError

    # Construct flow model
    model = nf.NormalizingFlow(base, flows)
    return model

def objective(sig, hx, hy, dm, dx, dy, reg):
    S = (1 + reg) * np.eye(dx) - sig @ sig.T
    B = hx - sig @ hy
    obj = 0.5 / np.log(2) * npla.slogdet(
        np.eye(dm) + hy.T @ hy + B.T @ la.solve(S, B)
    )[1]
    return obj

In [39]:
num_samples=10000
flow_iter=4000
n_flows=8

enable_cuda=True
device = torch.device('cuda' if torch.cuda.is_available() and enable_cuda else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [40]:
lamda_m, lamda_x, lamda_y, w_x1_vals, w_x2, w_y1, w_y2 = get_params()
dm = lamda_m.size  # Number of dimensions of M
dx = lamda_x.size  # Number of dimensions of X
dy = lamda_y.size  # Number of dimensions of Y

In [41]:
######### Prepare the pid flows #########
dim_mxy = dm + dx + dy
latent_dim = dim_mxy
base = dist.GaussianPCA(dim=dim_mxy, latent_dim=latent_dim, sigma=0.1)
flow = pid_flows(base, n_flows, latent_dim).to(device)

In [42]:
w_x1 = w_x1_vals[0]
w_x = np.array([[w_x1, w_x2]])  # Binomial thinning weights: shape (dx, dm)
w_y = np.array([[w_y1, w_y2]])

m, x, y = sample_mult_poisson(num_samples, lamda_m, w_x, w_y, lamda_x, lamda_y)  # Shape: (d, n)

In [43]:
######### Train pid flows #########
optimizer = torch.optim.Adam(flow.parameters(), lr=5e-4, weight_decay=1e-5)

for it in tqdm(range(flow_iter)):
    flow.train()
    optimizer.zero_grad()

    x_ = torch.from_numpy(x.T).float()
    y_ = torch.from_numpy(y.T).float()
    m_ = torch.from_numpy(m.T).float()

    mxy = torch.cat([m_, x_, y_], dim=-1).to(device)

    loss = flow.forward_kld(mxy)

    # Do backprop and optimizer step
    if ~(torch.isnan(loss) | torch.isinf(loss)):
        loss.backward()
        optimizer.step()

100%|██████████| 4000/4000 [04:03<00:00, 16.40it/s]


In [44]:
######### Get params for thin pid #########
cov_mxy = flow.q0.get_covariance(detach=True).cpu().numpy()
ret = pid.utils.whiten(cov_mxy, dm, dx, dy, ret_channel_params=True)
sig_mxy, hx, hy, hxy, sigxy = ret
print(f'sig_mxy:\n{sig_mxy}')
print(f'hx: {hx}, hy: {hy}')

sig, obj, _ = pid.optimizers.thinpid_exact_pid_minimizer(hx, hy, ret_obj=True, reg=1e-7)
union_info = objective(sig, hx, hy, dm, dx, dy, reg=1e-7)
print(f'union_info: {union_info}')

sig_mxy:
[[ 1.0000002e+00 -4.5562608e-08 -7.9683465e-01  6.3979053e-01]
 [ 5.6032402e-08  1.0000002e+00 -4.6272628e-04  1.8796425e-02]
 [-7.9683465e-01 -4.6272628e-04  1.6349455e+00  3.2115471e-01]
 [ 6.3979053e-01  1.8796425e-02  3.2115471e-01  1.4096851e+00]]
hx: [[-7.9683465e-01 -4.6272628e-04]], hy: [[0.63979053 0.01879643]]
union_info: 0.35530715737387863


In [45]:
from pid.optimizers import exact_gauss_tilde_pid, exact_tilde_union_info_minimizer
from pid.utils import whiten

mxy = np.vstack((m, x, y))
cov = np.corrcoef(mxy)    # Shape: (dm+dx+dy, dm+dx+dy)
print(cov)

ret = whiten(cov, dm, dx, dy, ret_channel_params=True)
sig_mxy, hx, hy, hxy, sigxy = ret
print(sig_mxy)
print(hx, hy)

sig, obj, _ = exact_tilde_union_info_minimizer(hx, hy, ret_obj=True, reg=1e-7)
union_info = objective(sig, hx, hy, dm, dx, dy, reg=1e-7)

print(union_info)

[[1.         0.0073672  0.00539724 0.40296943]
 [0.0073672  1.         0.50387965 0.41954064]
 [0.00539724 0.50387965 1.         0.20822479]
 [0.40296943 0.41954064 0.20822479 1.        ]]
[[ 1.00000000e+00 -1.47451996e-17  4.09968486e-03  4.92610757e-01]
 [-1.47454497e-17  1.00000000e+00  5.83336686e-01  5.13021169e-01]
 [ 4.09968486e-03  5.83336686e-01  1.34029850e+00  2.95818231e-01]
 [ 4.92610757e-01  5.13021169e-01  2.95818231e-01  1.50585608e+00]]
[[0.00409968 0.58333669]] [[0.49261076 0.51302117]]
0.43051307160023167
